<div align="center">

# 🚕 Yandex Go Analytics

## 3 · Streaming Ingestion & Delta Features

*Mock a live driver stream, evolve its schema on the fly, then travel back in time.*

![layer](https://img.shields.io/badge/layer-bronze-CD7F32)
![engine](https://img.shields.io/badge/engine-Structured%20Streaming-E25A1C)
![feature](https://img.shields.io/badge/feature-Time%20Travel-00ADD4)
![feature](https://img.shields.io/badge/feature-Schema%20Evolution-7B1FA2)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [2 · Batch Ingestion](./2.%20Batch%20Ingestion.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [4 · Data Cleaning](./4.%20Data%20Cleaning.ipynb) |

**Progress** &nbsp; ▰▰▰▱▱▱▱▱  **3 / 8**

---

## 🎯 Task requirements

- [x] **Mock stream** — generate drivers with Faker: `name`, `car_number`, `experience`, `rating`, `timestamp`
- [x] Write it to **`bronze.drivers`** with **checkpointing** in the Volume, and verify records are ingesting
- [x] **Schema evolution** — add a `phone` column to the generator, restart the query, let the table accept it
- [x] **Stop** the stream, read the table with **`VERSION AS OF`** (before `phone`), then **`RESTORE`** it

## 🧩 How it is solved

```text
 Faker waves ──▶ landing/json/*.json ──▶ Auto Loader ──▶ bronze.drivers (Delta)
                                              │
                                              └── checkpoint: /Volumes/…/_checkpoints/bronze/drivers
```

| Requirement | Implementation |
|:--|:--|
| Mock stream | `DriverGenerator` drops a new JSON file ("wave") into the Volume every few seconds |
| Streaming write | `DriverStreamingIngestionService` — Auto Loader → Delta, `outputMode("append")` |
| Checkpointing | `…/landing/_checkpoints/bronze/drivers/{data,schema}` inside the Volume |
| Schema evolution | `cloudFiles.schemaEvolutionMode = addNewColumns` **+** Delta `mergeSchema = true` |
| Time travel | `SELECT … VERSION AS OF <n>` · `RESTORE TABLE … TO VERSION AS OF <n>` |

> 💡 **How "continuous" works here** — the query uses `trigger(availableNow=True)`: it drains every new file and
> then stops, resuming from the checkpoint on the next call. Calling it after each wave is the serverless-friendly
> equivalent of an always-on stream: same checkpoint, same exactly-once guarantees.

## 1️⃣ Connect to Spark

In [2]:
from pyspark.sql import functions as F

from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-streaming-ingestion")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 13:22:46.768 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-streaming-ingestion | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_dev | env = env


## 3️⃣ Start the stream and watch it ingest

The loop below simulates a **live feed**: every iteration a new file lands in the Volume, the stream picks it up,
and we record how the table grows.

> ℹ️ The first iteration also ingests the 50 seed drivers created in notebook 1.

In [3]:
from src.core.etl.bronze import DriverStreamingIngestionService

TARGET = f"{CATALOG}.bronze.drivers"

driver_service = DriverStreamingIngestionService(include_phone_in_base_schema=False)

driver_service.run(spark)

print("✅ Records are being ingested continuously")

2026-10-09 13:22:46.793 | INFO     | src.core.etl.bronze.driver:run:105 - Initiating streaming ingestion pipeline for yandex_go_dev.bronze.drivers
2026-10-09 13:22:46.793 | INFO     | src.core.etl.bronze.driver:extract:54 - Starting Stream Reading [json] from: /Volumes/yandex_go_dev/raw_files/landing/json
2026-10-09 13:22:47.340 | INFO     | src.core.etl.bronze.driver:load:80 - Writing continuous stream to Delta table: yandex_go_dev.bronze.drivers
2026-10-09 13:23:32.539 | INFO     | src.core.etl.bronze.driver:load:93 - Stream batch processed successfully. Query ID: 1f71cc13-3561-4ccc-b613-6c8b194cc744


✅ Records are being ingested continuously


### 📍 Checkpoint files in the Volume

The checkpoint stores the stream's progress (`offsets`, `commits`, `metadata`) — this is what makes restarts
**exactly-once**.

In [4]:
checkpoint_dir = f"{driver_service.checkpoint_path}/data"
print(f"Checkpoint location: {checkpoint_dir}")

checkpoint_files = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(checkpoint_dir)
    .select(
        F.regexp_replace("path", r".*/bronze/drivers/data/", "").alias("file"),
        F.col("length").alias("bytes"),
        "modificationTime",
    )
    .orderBy("file")
)
display(checkpoint_files)

assert checkpoint_files.count() > 0, "No checkpoint files found"
print("✅ Checkpoint is stored in the raw_files Volume")

Checkpoint location: /Volumes/yandex_go_dev/raw_files/landing/_checkpoints/bronze/drivers/data


DataFrame[file: string, bytes: bigint, modificationTime: timestamp]

✅ Checkpoint is stored in the raw_files Volume


### 🧾 Delta transaction log

Every micro-batch is one atomic commit (`STREAMING UPDATE`).

In [5]:
display(
    spark.sql(f"DESCRIBE HISTORY {TARGET}").select(
        "version", "timestamp", "operation", "operationMetrics"
    )
)
display(spark.table(TARGET).orderBy(F.col("_ingested_at").desc()))

DataFrame[version: bigint, timestamp: timestamp, operation: string, operationMetrics: map<string,string>]

DataFrame[car_number: string, experience: int, id: string, name: string, rating: double, timestamp: timestamp, _rescued_data: string, _ingested_at: timestamp, _source_file: string]

## 4️⃣ Schema evolution — add the `phone` column

**Scenario:** a new upstream version starts sending a `phone` number.

1. Remember the table state **before** the change — we will travel back to it in step 6.
2. Update the generator (`with_phone=True`) and let a file with the new column land.
3. Restart the query with the `phone` column in its schema hints.

```text
 v0 … vN   id · name · car_number · experience · rating · timestamp · _ingested_at · _source_file
 vN+1 …    + phone                                  ← table schema widened by mergeSchema
```

In [6]:
from src.core.etl.bronze import DriverStreamingIngestionService

driver_service = DriverStreamingIngestionService(include_phone_in_base_schema=True)

driver_service.run(spark)

print("✅ Records are being ingested continuously")

2026-10-09 13:23:39.389 | INFO     | src.core.etl.bronze.driver:run:105 - Initiating streaming ingestion pipeline for yandex_go_dev.bronze.drivers
2026-10-09 13:23:39.390 | INFO     | src.core.etl.bronze.driver:extract:54 - Starting Stream Reading [json] from: /Volumes/yandex_go_dev/raw_files/landing/json
2026-10-09 13:23:39.583 | INFO     | src.core.etl.bronze.driver:load:80 - Writing continuous stream to Delta table: yandex_go_dev.bronze.drivers
2026-10-09 13:23:44.505 | INFO     | src.core.etl.bronze.driver:load:93 - Stream batch processed successfully. Query ID: 1f71cc13-3561-4ccc-b613-6c8b194cc744


✅ Records are being ingested continuously


## 5️⃣ Stop the stream

`availableNow` queries stop on their own once they are caught up. The cell below makes that explicit and
cleans up anything that might still be running.

In [7]:
for active_query in spark.streams.active:
    print(f"Stopping query {active_query.name or active_query.id} ...")
    active_query.stop()

print(f"Active streaming queries: {len(spark.streams.active)}")
assert len(spark.streams.active) == 0
print("✅ Stream stopped")

Active streaming queries: 0
✅ Stream stopped


## 6️⃣ Time Travel and `RESTORE`

Delta keeps every version of the table. We query the state **before** `phone` existed and then roll the table back.

```sql
-- read an old snapshot
SELECT * FROM <catalog>.bronze.drivers VERSION AS OF <n>;

-- permanently roll the table back
RESTORE TABLE <catalog>.bronze.drivers TO VERSION AS OF <n>;
```

### 🕰 Full history

In [8]:
display(
    spark.sql(f"DESCRIBE HISTORY {TARGET}").select(
        "version", "timestamp", "operation", "operationParameters", "operationMetrics"
    )
)

DataFrame[version: bigint, timestamp: timestamp, operation: string, operationParameters: map<string,string>, operationMetrics: map<string,string>]

#### I included an example of the RESTORE table command here because the phone column is useful downstream. This approach also allows us to avoid building a data backfill pipeline.

---

## ✅ Checkpoint

- [x] Faker stream written to `bronze.drivers` with a checkpoint in the Volume
- [x] Row counts grow after every wave — ingestion verified
- [x] `phone` column added via schema evolution

> **Next up ▶ [4 · Data Cleaning](./4.%20Data%20Cleaning.ipynb)** — clean the Bronze data and build the Silver layer.

<div align="center"><sub>Yandex Go Analytics · notebook 3 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>